In [1]:
import pandas as pd

In [4]:
source1 = pd.read_csv(
    '../../student_resource/dataset/train/train_source1.tsv',
    sep="\t"
)

source2 = pd.read_csv(
    '../../student_resource/dataset/train/train_source2.tsv',
    sep="\t"
)

source3 = pd.read_csv(
    '../../student_resource/dataset/train/train_source3.tsv',
    sep="\t"
)

In [5]:
display(source1.head())
display(source2.head())
display(source3.head())

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India


In [6]:
summary = pd.DataFrame({
    "source": ["Source 1", "Source 2", "Source 3"],
    "rows": [
        len(source1),
        len(source2),
        len(source3)
    ],
    "columns": [
        source1.shape[1],
        source2.shape[1],
        source3.shape[1]
    ]
})

display(summary)

,source,rows,columns
0,Source 1,2206821,4
1,Source 2,5034616,4
2,Source 3,5285603,4


In [7]:
def missing_report(df, name):
    report = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": df.isna().mean() * 100
    })
    
    print(f"\n===== {name} =====")
    display(report)

missing_report(source1, "SOURCE 1")
missing_report(source2, "SOURCE 2")
missing_report(source3, "SOURCE 3")


===== SOURCE 1 =====


,missing_count,missing_percent
entity_id,0,0.0
business_name,0,0.0
business_address,0,0.0
country,0,0.0



===== SOURCE 2 =====


,missing_count,missing_percent
entity_id,0,0.000000
business_name,2,0.000040
business_address,168967,3.356105
country,0,0.000000



===== SOURCE 3 =====


,missing_count,missing_percent
entity_id,0,0.000000
business_name,13,0.000246
business_address,175916,3.328211
country,0,0.000000


Duplicate entity IDs

In [8]:
for name, df in [
    ("Source 1", source1),
    ("Source 2", source2),
    ("Source 3", source3)
]:
    print(f"\n{name}")
    print("Duplicate entity IDs:", df["entity_id"].duplicated().sum())


Source 1
Duplicate entity IDs: 0

Source 2
Duplicate entity IDs: 0

Source 3
Duplicate entity IDs: 0


whether multiple records have exactly the same name/address.

In [9]:
for name, df in [
    ("Source 1", source1),
    ("Source 2", source2),
    ("Source 3", source3)
]:
    duplicate_pairs = df.duplicated(
        subset=["business_name", "business_address"],
        keep=False
    ).sum()

    print(f"{name}: {duplicate_pairs} rows involved in exact name+address duplicates")

Source 1: 0 rows involved in exact name+address duplicates
Source 2: 50969 rows involved in exact name+address duplicates
Source 3: 37283 rows involved in exact name+address duplicates


Because country may be useful for blocking:

In [10]:
for name, df in [
    ("Source 1", source1),
    ("Source 2", source2),
    ("Source 3", source3)
]:
    print(f"\n===== {name} =====")
    print(df["country"].value_counts(dropna=False))


===== Source 1 =====
country
US       1323633
India     883188
Name: count, dtype: int64

===== Source 2 =====
country
US       3016817
India    2017799
Name: count, dtype: int64

===== Source 3 =====
country
US       3170056
India    2115547
Name: count, dtype: int64


This is useful for deciding how character n-grams will behave.

In [11]:
for df in [source1, source2, source3]:
    df["name_length"] = df["business_name"].fillna("").astype(str).str.len()
    df["address_length"] = df["business_address"].fillna("").astype(str).str.len()

In [12]:
for name, df in [
    ("Source 1", source1),
    ("Source 2", source2),
    ("Source 3", source3)
]:
    print(f"\n===== {name} =====")
    
    display(
        df[
            ["name_length", "address_length"]
        ].describe()
    )


===== Source 1 =====


,name_length,address_length
count,2.206821e+06,2.206821e+06
mean,2.403440e+01,5.206621e+01
std,7.740533e+00,2.532972e+01
min,3.000000e+00,1.100000e+01
25%,1.800000e+01,3.300000e+01
50%,2.400000e+01,4.100000e+01
75%,3.000000e+01,7.000000e+01
max,1.050000e+02,2.560000e+02



===== Source 2 =====


,name_length,address_length
count,5.034616e+06,5.034616e+06
mean,2.510357e+01,4.622591e+01
std,8.894694e+00,2.484467e+01
min,0.000000e+00,0.000000e+00
25%,1.900000e+01,3.000000e+01
50%,2.500000e+01,3.700000e+01
75%,3.100000e+01,6.100000e+01
max,1.040000e+02,2.490000e+02



===== Source 3 =====


,name_length,address_length
count,5.285603e+06,5.285603e+06
mean,2.520214e+01,4.671444e+01
std,9.493635e+00,2.164981e+01
min,0.000000e+00,0.000000e+00
25%,1.800000e+01,3.500000e+01
50%,2.500000e+01,4.200000e+01
75%,3.100000e+01,5.400000e+01
max,1.230000e+02,2.400000e+02


Look at actual records

In [13]:
print("SOURCE 1")
display(
    source1[
        ["entity_id", "business_name", "business_address", "country"]
    ].sample(10, random_state=42)
)

print("SOURCE 2")
display(
    source2[
        ["entity_id", "business_name", "business_address", "country"]
    ].sample(10, random_state=42)
)

print("SOURCE 3")
display(
    source3[
        ["entity_id", "business_name", "business_address", "country"]
    ].sample(10, random_state=42)
)

SOURCE 1


,entity_id,business_name,business_address,country
2195840,S1-53356671,Pediatric Medicine PLLC,"4850 20, Otisco, NY",US
1487051,S1-320151505,Fetech National Twin,"19034 Woodburn Road, Woodburn, IN",US
1878277,S1-938947364,General Design Innovations LLC,"7241 Osage Avenue, Mesa, AZ",US
993966,S1-195839862,Construction Ideaz Papers Private Limited,"Building No.4/606, Prabhul Cottage Karimbalur,...",India
565428,S1-655046555,Penaloza and Bittle First Inc.,"4255 Charleswood Avenue, Memphis, TN",US
1645753,S1-758070915,National Investments LLC,"1641 Virginia Lane, Hueytown, AL",US
1165724,S1-169338169,Reus Idaho Company,"Sanford, ME, 31 Guillemette Street",US
363842,S1-796421498,Creative Projects Limited,"12Thfloor, C, 1204, Roya Oasis, Jankalyan Naga...",India
962445,S1-802535179,Goar Stone LLC,"2105 Wood Ridge Cove, Cedar Park, TX",US
964820,S1-97176033,Lakshmi Consultants Private Limited,"A-301, New Sai Dham Chsl, Ramdev Park Road, Th...",India


SOURCE 2


,entity_id,business_name,business_address,country
2641297,S2-61703745,Harbor Center,"6309 EVANGELINE TRAIL, AUSTIN, TX",US
1856454,S2-420680068,STEWARD ACE PREMIER,"1029 MAPLE HILL RD, LEBANON, TN",US
371269,S2-190917211,Klapper and Lott,"368 VINE STREET, TOOELE, UT",US
793182,S2-109564077,EAR NOSE & THROAT CARE,"YORK RD, LUTHERVILLE, MD",US
4159519,S2-483058783,Shield Inrfabuild Private Limited,"206 FLAT NO. LG 1 KH NO. 514, 516 SHYAM BHAWAN...",India
2290871,S2-257746699,#holtline,"HOUSTON, 6119 DARLINGHURST DRIVE, TX",US
2479261,S2-476138086,Cancer [Services],NaN,US
606462,S2-420894180,Smt Hermes Private Limited Center,"Delhi, HOUSE NO 138, 2ND FLOOR, BLK C, PKT 2 D...",India
3805434,S2-498993768,AMARDEEP PTER PRIVATE LIMITED,"332NEAR TELCO SERVICE STATION RANGPURI, NEW DE...",India
2960649,S2-54193377,Impex-+-Brothers,"FLAT B 1801, PL-15&17, SECTOR 7, PATEL HERITAG...",India


SOURCE 3


,entity_id,business_name,business_address,country
726818,S3-239979459,Dr Apex Pvt Ltd Partners,"75 Tulshibaugwale Colonysahakarnagar, Pune, MH",India
1045360,S3-109491785,Shiva Mines Pvt Ltd,"F-17, Sector 22, Noida, Gautam Buddha Nagar, उ...",India
4785624,S3-586835953,Vanguard Métropolitan Sunshine,"3518-B Crestview Ln, Catoosa, Oklahoma",US
2503843,S3-449000694,Vanguard Apoge-Inc,"2150 350, Greencastle, Indiana",US
3149825,S3-397034729,रियल एग्रो प्राइवेट लिमिटेड,"H.no 829 D-225vivek Vihar, Delhi, दिल्ली",India
2749542,S3-553748449,Irinovi,"324 Laramie Ave, Chicago, Illinois",US
3399114,S3-787185173,Jarleon Worldwide Llc,"353 Spruce Pine Road, Abingdon, Maryland",US
1611580,S3-691178620,Sfoasseeg L.L.C.,"1161d Brookside Ave, Evansdale, Iowa",US
3556867,S3-138902486,Brixhalo t/a Creative Enterprises,"6, Madurai Road, Trichy, Tamil Nadu",India
4340188,S3-568450939,Nachiket Solutions Limited,"138 / 6, Kiran Complex Zone - Ii, M. P. Nagar ...",India


Concatinate all

In [14]:
all_data = pd.concat(
    [source1, source2, source3],
    ignore_index=True
)

In [15]:
all_data.shape

(12527040, 6)

Ground Truth

In [20]:
ground_truth = pd.read_csv('../../student_resource/dataset/train/train_ground_truth.tsv',
    sep="\t")

In [21]:
def split_matches(x):
    if pd.isna(x) or str(x).strip() == "":
        return []
    
    return [
        item.strip()
        for item in str(x).split(",")
        if item.strip()
    ]

ground_truth["matched_ids_list"] = (
    ground_truth["matched_entity_ids"]
    .apply(split_matches)
)

In [22]:
ground_truth["num_matches"] = (
    ground_truth["matched_ids_list"].str.len()
)

In [23]:
display(
    ground_truth[
        [
            "source1_entity_id",
            "num_matches",
            "matched_ids_list"
        ]
    ].head(10)
)

,source1_entity_id,num_matches,matched_ids_list
0,S1-965667,5,"[S2-681193310, S2-743505751, S3-775321672, S3-..."
1,S1-55344266,4,"[S2-249013014, S2-197070651, S3-478195123, S3-..."
2,S1-343815751,3,"[S2-790675320, S2-479876582, S3-878454467]"
3,S1-656753428,3,"[S2-153058913, S2-24659151, S3-679606215]"
4,S1-102811957,6,"[S2-478959098, S2-553508714, S2-625774905, S3-..."
5,S1-18727616,4,"[S2-755677256, S3-187831601, S3-641489370, S3-..."
6,S1-318373630,2,"[S2-660036492, S3-804600254]"
7,S1-86989137,2,"[S3-274817120, S3-312496301]"
8,S1-29845983,2,"[S2-648035184, S3-588502663]"
9,S1-789009573,3,"[S2-383871912, S3-74481402, S3-576451439]"


In [24]:
print(
    ground_truth["num_matches"].describe()
)

count    2.206821e+06
mean     3.461253e+00
std      1.705323e+00
min      0.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: num_matches, dtype: float64


how many S1 entities have ground truth?

In [25]:
num_s1 = len(source1)

num_gt_s1 = ground_truth["source1_entity_id"].nunique()

print("Total S1 records:", num_s1)
print("S1 records appearing in ground truth:", num_gt_s1)
print(
    "S1 records NOT appearing in ground truth:",
    num_s1 - num_gt_s1
)

Total S1 records: 2206821
S1 records appearing in ground truth: 2206821
S1 records NOT appearing in ground truth: 0


Count S2/S3 matches separately

In [26]:
from collections import Counter

match_counter = Counter()

for ids in ground_truth["matched_ids_list"]:
    for entity_id in ids:
        if entity_id.startswith("S2-"):
            match_counter["S2"] += 1
        elif entity_id.startswith("S3-"):
            match_counter["S3"] += 1

print(match_counter)

Counter({'S3': 3944746, 'S2': 3693619})


Verify ground truth IDs actually exist

In [27]:
all_ids = set(all_data["entity_id"])

missing_gt_ids = []

for ids in ground_truth["matched_ids_list"]:
    for entity_id in ids:
        if entity_id not in all_ids:
            missing_gt_ids.append(entity_id)

print("Ground-truth IDs missing from source data:", len(missing_gt_ids))

if missing_gt_ids:
    print(missing_gt_ids[:20])

Ground-truth IDs missing from source data: 0


In [28]:
import re
import unicodedata


def normalize_text(text):
    """
    Basic normalization:
    - handle missing values
    - Unicode normalization
    - lowercase
    - replace punctuation/special characters with spaces
    - collapse multiple spaces
    """
    
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)
    
    # Lowercase
    text = text.lower()
    
    # Replace punctuation/symbols with space
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    
    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

Apply normalization

In [29]:
for df in [source1, source2, source3]:
    
    df["norm_name"] = df["business_name"].fillna("").map(normalize_text)
    df["norm_address"] = df["business_address"].fillna("").map(normalize_text)